In [1]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 19.1 MB/s eta 0:00:00


# DATA

### **Cleaning**

In [2]:
!cp -r drive/MyDrive/Bible\ Project temp/

In [3]:
import fitz
import re

EFIK_PATH = "temp/Efik-All-Bible.pdf"
NIV_PATH = "temp/NIV-Bible.pdf"

In [4]:
import fitz  # PyMuPDF
import re

def get_all_verses(*pdf_paths):
    all_results = []
    verse_start_pattern = re.compile(r'^(\d+)\s*(.*)', re.MULTILINE)

    for path in pdf_paths:
        try:
            doc = fitz.open(path)
            file_verses = []
            current_verse = ""

            for page in doc:
                text = page.get_text()
                lines = text.splitlines()

                for line in lines:
                    line = line.strip()
                    if not line: continue

                    # Check if line starts with a verse number
                    match = verse_start_pattern.match(line)

                    if match:
                        # If we were already building a verse, save it before starting new one
                        if current_verse:
                            file_verses.append(current_verse)
                        current_verse = line
                    else:
                        # If it doesn't start with a digit, it's a continuation of the previous verse
                        if current_verse:
                            current_verse += " " + line

            # Append the very last verse of the file
            if current_verse:
                file_verses.append(current_verse)

            all_results.append(file_verses)
            doc.close()

        except Exception as e:
            print(f"Error loading {path}: {e}")
            all_results.append([]) # Keep the list index consistent even on failure

    return all_results



In [140]:
efik, english = get_all_verses(EFIK_PATH, NIV_PATH)

In [141]:
efik = [ef for ef in efik if "Bible Society of Nigeria" not in ef][83:]
english = english[17:]

In [142]:
def get_all_chapter_1s_efik(efik_list):
  chapter_1_indices = []
  for i, verse in enumerate(efik_list):
    # Check if the verse starts with '1' immediately followed by a letter, and has a length greater than 10 characters
    if re.match(r'^1[a-zA-Z]', verse) and len(verse) > 10:
      chapter_1_indices.append(i)
  return chapter_1_indices

In [143]:
ids = get_all_chapter_1s_efik(efik)

In [144]:
efik_verse_gaps = [ids[i+1] - ids[i] for i in range(len(ids)-1)]

In [145]:
import re

def get_leading_number(verse_string):
    """
    Extracts the integer value of leading digits from a verse string.
    Returns None if no leading digits are found.
    """
    match = re.match(r'^(\d+)', verse_string)
    if match:
        return int(match.group(1))
    return None

def get_all_chapter_1s_english(eng_list):
  chapter_1_indices = []
  for i in range(len(eng_list) - 1):
    current_verse = eng_list[i]
    next_verse = eng_list[i+1]

    try:
      upper_verse = eng_list[i+2]
    except IndexError as e:
      pass

    current_leading_number = get_leading_number(current_verse)
    next_leading_number = get_leading_number(next_verse)
    upper_leading_number = get_leading_number(upper_verse)

    # Chapter 1 is labelled 2
    if next_leading_number == 2 and upper_leading_number == 2 :
      chapter_1_indices.append(i+1)
      continue

    # Chapter 1 is labelled 1
    if next_leading_number == 2 and current_leading_number == 1:
      chapter_1_indices.append(i)

    # Chapter 1 is labelled any other number
    if next_leading_number == 2 and upper_leading_number > 2:
      chapter_1_indices.append(i)


  return chapter_1_indices

In [160]:
eng_ids = get_all_chapter_1s_english(english)

In [116]:
len(eng_ids)

1250

In [149]:
def remove_duplicates_preserve_order(input_list):
    unique_list = []
    seen_elements = set()
    for item in input_list:
        if item not in seen_elements:
            unique_list.append(item)
            seen_elements.add(item)
    return unique_list

eng_ids = remove_duplicates_preserve_order(eng_ids)
print("Duplicates removed from eng_ids while preserving order.")
print(f"New length of eng_ids: {len(eng_ids)}")

Duplicates removed from eng_ids while preserving order.
New length of eng_ids: 1190


In [159]:
english[eng_ids[4]]

'3Now the serpent was more crafty than any of the wild animals the Lord God had made. He said to the woman, "Did God really say, \'You must not eat from any tree in the garden\'?"'

In [165]:
eng_ids = [eng_ids[i] for i in range(len(eng_ids)-1) if eng_ids[i] != eng_ids[i+1]]

In [167]:
english_verse_gaps = [eng_ids[i+1] - eng_ids[i] for i in range(len(eng_ids)-1)]

In [170]:
efik_verse_gaps[:15]

[31, 25, 24, 26, 32, 22, 24, 22, 29, 32, 32, 20, 18, 24, 21]

In [171]:
english_verse_gaps[:15]

[31, 25, 24, 26, 35, 23, 24, 22, 29, 32, 33, 20, 18, 24, 21]